# Klasifikasi Naive Bayes Manual (Kelompok 3)

**Mata Kuliah:** Machine Learning (Kelas C), Bapak Adi Purnawan

**Anggota:** Deliana Br Manalu (2305551036) · Ravi Arnan Irianto (2305551076) · Ezza Putra Wibawa (2305551144) · Devin (2305551173)

---

Notebook ini mengimplementasikan **Gaussian Naive Bayes** dari nol, tanpa
scikit-learn. Model dan metriknya dihitung sendiri. Datanya dibaca dari berkas bersih
hasil notebook `02_bersihkan_data`, jadi tidak ada library machine learning yang dipakai.

Caranya: hitung mean dan std tiap fitur per kelas, lalu hitung probabilitas tiap kelas
dengan rumus Gaussian PDF dan pilih yang tertinggi.

Dataset yang dipakai: Stroke Prediction versi bersih dari notebook `02_bersihkan_data`
(5.109 baris). Hanya kolom numerik yang dipakai: age, hypertension, heart_disease,
avg_glucose_level, bmi. Target: stroke.

Pasangan notebook ini untuk KNN ada di `03a_knn_manual.ipynb`.
Versi scikit-learn menyusul di `04_model_library.ipynb`.


## 1. Muat Dataset

Data diambil langsung dari **berkas dataset bersih** yang dihasilkan notebook
`02_bersihkan_data`, yaitu `artifacts/stroke_bersih.csv` (5.109 baris, `bmi` sudah diisi
prediksi Ridge, `gender` "Other" sudah dibuang).

Karena pembersihannya sudah selesai di notebook `02`, notebook ini bisa fokus penuh ke
algoritmanya. Kolom `gender` tetap dimuat karena dibutuhkan skor medis di bagian 5.

In [ ]:
import pandas as pd
import numpy as np
import math
import random
import matplotlib.pyplot as plt

random.seed(42)
np.random.seed(42)

# Dataset bersih hasil notebook 02_bersihkan_data, diambil dari repositori supaya
# isinya sama persis dengan yang dipakai notebook lain.
URL_BERSIH = ("https://raw.githubusercontent.com/ravi-arnan/machine-learning/"
              "main/artifacts/stroke_bersih.csv")

KOLOM = ["gender", "age", "hypertension", "heart_disease", "avg_glucose_level", "bmi", "stroke"]
df = pd.read_csv(URL_BERSIH)[KOLOM]

# Fitur untuk model. gender tidak dipakai sebagai fitur, hanya untuk skor medis di bagian 5.
fitur = ["age", "hypertension", "heart_disease", "avg_glucose_level", "bmi"]

print("Ukuran dataset bersih:", df.shape)
print("BMI kosong:", df["bmi"].isna().sum())
print("5 baris pertama:")
df.head()

## 2. Train-Test Split Manual

Split 80:20 tanpa library. Acak indeks dulu, lalu potong. Karena datanya tidak
seimbang (hanya 4,87% stroke), proporsi kelas di train dan test dicetak supaya
kelihatan apakah kedua bagian kebagian kasus stroke.

In [ ]:
def train_test_split_manual(data, test_size=0.2, seed=42):
    """Bagi baris data jadi train dan test secara manual (80:20)."""
    random.seed(seed)
    n = len(data)
    n_test = int(n * test_size)

    indices = list(range(n))
    random.shuffle(indices)

    test_idx = indices[:n_test]
    train_idx = indices[n_test:]

    return data.iloc[train_idx], data.iloc[test_idx]


latih, uji = train_test_split_manual(df, test_size=0.2, seed=42)

X_train, y_train = latih[fitur].values, latih["stroke"].values
X_test, y_test = uji[fitur].values, uji["stroke"].values

print(f"Train: {len(latih)} baris")
print(f"Test:  {len(uji)} baris")
print(f"Proporsi stroke di train: {y_train.mean():.3f}")
print(f"Proporsi stroke di test:  {y_test.mean():.3f}")

## 3. Fungsi Evaluasi Manual

Semua metrik dihitung dari confusion matrix tanpa library.

In [ ]:
def confusion_matrix_manual(y_true, y_pred):
    """Hitung TP, TN, FP, FN manual."""
    tp = tn = fp = fn = 0
    for t, p in zip(y_true, y_pred):
        if t == 1 and p == 1:
            tp += 1
        elif t == 0 and p == 0:
            tn += 1
        elif t == 0 and p == 1:
            fp += 1
        elif t == 1 and p == 0:
            fn += 1
    return tp, tn, fp, fn


def hitung_metrik(y_true, y_pred):
    """Hitung akurasi, presisi, recall, F1 manual."""
    tp, tn, fp, fn = confusion_matrix_manual(y_true, y_pred)

    akurasi = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    presisi = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * presisi * recall / (presisi + recall) if (presisi + recall) > 0 else 0

    return {
        "akurasi": round(akurasi, 4),
        "presisi": round(presisi, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }

## 4. Gaussian Naive Bayes Manual

Hitung mean dan std tiap fitur per kelas, lalu hitung probabilitas dengan rumus
Gaussian PDF. Pilih kelas dengan probabilitas tertinggi.

Perhitungan dikerjakan dalam bentuk log supaya angka sangat kecil tidak langsung
menjadi nol karena perkalian berulang.

In [ ]:
def gaussian_pdf(x, mean, std):
    """Hitung probabilitas Gaussian."""
    if std == 0:
        return 1.0 if x == mean else 0.0001
    eksponen = math.exp(-((x - mean) ** 2) / (2 * std ** 2))
    return (1 / (math.sqrt(2 * math.pi) * std)) * eksponen


class NaiveBayesManual:
    """Gaussian Naive Bayes buatan sendiri."""

    def fit(self, X, y):
        n, m = X.shape
        self.kelas = sorted(set(y))
        self.prior = {}
        self.mean = {}
        self.std = {}

        for c in self.kelas:
            Xc = X[y == c]
            self.prior[c] = len(Xc) / n
            self.mean[c] = Xc.mean(axis=0)
            self.std[c] = Xc.std(axis=0)

    def predict(self, X):
        y_pred = []
        for titik in X:
            prob_kelas = {}
            for c in self.kelas:
                prob = math.log(self.prior[c])
                for i in range(len(titik)):
                    prob += math.log(gaussian_pdf(titik[i], self.mean[c][i], self.std[c][i]) + 1e-10)
                prob_kelas[c] = prob
            y_pred.append(max(prob_kelas, key=prob_kelas.get))
        return np.array(y_pred)


nb = NaiveBayesManual()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)
print("Naive Bayes selesai.")

In [ ]:
metrik_nb = hitung_metrik(y_test, y_pred_nb)

print("=" * 50)
print("HASIL NAIVE BAYES")
print("=" * 50)
print(f"  Akurasi:  {metrik_nb['akurasi']}")
print(f"  Presisi:  {metrik_nb['presisi']}")
print(f"  Recall:   {metrik_nb['recall']}")
print(f"  F1-score: {metrik_nb['f1']}")
print()
print("Confusion Matrix:")
print(f"  TP={metrik_nb['tp']}  FP={metrik_nb['fp']}")
print(f"  FN={metrik_nb['fn']}  TN={metrik_nb['tn']}")

### 4a. Lihat Parameternya

Naive Bayes mudah diperiksa karena parameternya cuma mean dan std tiap fitur per
kelas. Berguna untuk memastikan sesuai akal sehat.

In [ ]:
nama_fitur = ["age", "hypertension", "heart_disease", "avg_glucose_level", "bmi"]

for c in nb.kelas:
    label = "stroke" if c == 1 else "tidak stroke"
    print(f"Kelas {c} ({label}), prior {nb.prior[c]:.4f}")
    for i, nama in enumerate(nama_fitur):
        print(f"  {nama:18s} mean={nb.mean[c][i]:8.2f}  std={nb.std[c][i]:8.2f}")
    print()

## 5. Pembanding: Skor Risiko Stroke Klinis (CHA2DS2-VASc)

Sebelum ada machine learning, dunia medis sudah punya cara menghitung risiko stroke.
Salah satu yang paling dikenal adalah **CHA2DS2-VASc**, skor yang dipakai untuk
memperkirakan risiko stroke pada pasien fibrilasi atrium.

Skornya hanyalah penjumlahan poin:

$$\text{Skor} = C + H + A_2 + D + S_2 + V + A + Sc$$

dengan tiap komponen bernilai:

| Lambang | Arti | Poin |
|---|---|---|
| $C$ | Congestive heart failure (gagal jantung) | 1 |
| $H$ | Hypertension (hipertensi) | 1 |
| $A_2$ | Age $\geq$ 75 tahun | 2 |
| $D$ | Diabetes mellitus | 1 |
| $S_2$ | Stroke atau TIA sebelumnya | 2 |
| $V$ | Vascular disease (penyakit vaskular) | 1 |
| $A$ | Age 65 sampai 74 tahun | 1 |
| $Sc$ | Sex category (perempuan) | 1 |

### Adaptasi ke dataset kami

Dua komponen tidak bisa dipakai, dan ini harus disebut apa adanya:

- $S_2$ (riwayat stroke) justru **label yang mau diprediksi**, jadi tidak boleh jadi fitur.
- $V$ (penyakit vaskular) tidak ada di dataset.

Sisanya dipakai apa adanya, dengan dua proksi yang juga harus disebut:

- $C$ memakai kolom `heart_disease`
- $D$ memakai `avg_glucose_level >= 126` mg/dL, ambang diabetes menurut ADA

Karena itu skor maksimum di sini **6**, bukan 9.

Skor ini tidak dilatih dan tidak belajar dari data, ia aturan tetap. Justru itu gunanya
di sini: jadi pembanding "cara dokter" terhadap "cara machine learning".

In [ ]:
def skor_cha2ds2_vasc(baris):
    """CHA2DS2-VASc versi adaptasi. Penjelasan rumusnya ada di sel atas."""
    skor = 0
    if baris["heart_disease"] == 1:          # C, proksi gagal jantung
        skor += 1
    if baris["hypertension"] == 1:           # H
        skor += 1
    if baris["age"] >= 75:                   # A2
        skor += 2
    elif baris["age"] >= 65:                 # A
        skor += 1
    if baris["avg_glucose_level"] >= 126:    # D, ambang diabetes menurut ADA
        skor += 1
    if baris["gender"] == "Female":          # Sc
        skor += 1
    # S2 (riwayat stroke) sengaja tidak dipakai: itu label yang sedang diprediksi.
    # V (penyakit vaskular) tidak tersedia di dataset.
    return skor


skor = df.apply(skor_cha2ds2_vasc, axis=1)
skor_uji = skor.loc[uji.index].values

print("Sebaran skor pada data uji:")
print(pd.Series(skor_uji).value_counts().sort_index().to_string())

In [ ]:
baris_skor = []
for ambang in [1, 2, 3, 4]:
    pred = (skor_uji >= ambang).astype(int)
    m = hitung_metrik(y_test, pred)
    baris_skor.append({
        "Aturan": f"skor >= {ambang}",
        "akurasi": m["akurasi"],
        "presisi": m["presisi"],
        "recall": m["recall"],
        "f1": m["f1"],
        "ditandai": int(pred.sum()),
    })

pd.DataFrame(baris_skor)

## 6. Uji Kombinasi Atribut terhadap Confidence Level

Arahan dosen pada Pertemuan 3 meminta menentukan **fitur utama** dan **kombinasi atribut
terbaik untuk confidence level, bukan akurasi**. Notebook `13` sudah menjawabnya memakai
Logistic Regression. Di sini hal yang sama diulang memakai **Naive Bayes manual**:
delapan kombinasi atribut diadu, lalu diukur confidence level-nya.

**Kenapa precision, bukan akurasi.** Hanya 4,87% pasien yang benar-benar stroke, jadi
akurasi menyesatkan (bagian sebelumnya sudah memperlihatkannya). Confidence di sini
diukur dengan **precision**, dan satu angka ringkasannya **average precision (AP)**,
yaitu luas di bawah kurva precision-recall. Dicatat juga precision pada recall 0,80,
titik kerja yang dipakai notebook `07`.

**Protokol yang adil.** Semua kombinasi diuji pada baris dan lipatan yang persis sama:
lima lipatan stratified, skor diambil out-of-fold (tiap pasien dinilai oleh model yang
tidak pernah melihatnya saat latih). Setiap AP diberi selang kepercayaan bootstrap 95%.
Protokolnya meniru notebook `13`; lipatannya dibangun manual, jadi angkanya boleh
berbeda tipis dari jalur scikit-learn di sana.


### 6a. Menyusun delapan kombinasi atribut

Mesin uji ini memakai seluruh kolom data bersih, bukan hanya lima kolom numerik di bagian
awal, karena kombinasinya mencakup atribut sosial (pekerjaan, wilayah, status merokok).
Kolom kategorikal diubah menjadi one-hot.


In [ ]:
df_penuh = pd.read_csv(URL_BERSIH)

y_komb = df_penuh["stroke"].astype(int).to_numpy()
X_dasar = pd.get_dummies(df_penuh.drop(columns=["stroke"]), drop_first=True).astype(float)

print("Ukuran data bersih   :", df_penuh.shape)
print(f"Kasus stroke         : {int(y_komb.sum())} ({y_komb.mean()*100:.2f}%)")
print(f"Fitur dasar (one-hot): {X_dasar.shape[1]}")


Fitur turunan if-else dari laporan Pertemuan 3 ikut diuji supaya aturannya terukur,
bukan hanya diklaim. Nilainya ordinal, makin besar makin berisiko.


In [ ]:
def fitur_turunan(d):
    """Fitur turunan if-else, disalin dari laporan Pertemuan 3. Nilainya ordinal:
    semakin besar semakin berisiko."""
    f = pd.DataFrame(index=d.index)
    f["glukosa_kat"] = np.digitize(d["avg_glucose_level"], [100, 140, 200]).astype(float)
    f["bmi_kat"] = np.digitize(d["bmi"], [18.5, 25, 30]).astype(float)
    f["jumlah_faktor"] = (d["hypertension"] + d["heart_disease"]
                          + (d["avg_glucose_level"] > 200).astype(int)
                          + (d["bmi"] >= 30).astype(int)
                          + (d["age"] >= 60).astype(int)).astype(float)
    f["risiko_nested"] = 0.0
    f.loc[(d.hypertension == 1) | (d.heart_disease == 1), "risiko_nested"] = 1
    f.loc[(d.hypertension == 1) & (d.heart_disease == 1), "risiko_nested"] = 2
    f.loc[(d.hypertension == 1) & (d.heart_disease == 1)
          & (d.avg_glucose_level > 200), "risiko_nested"] = 3
    f.loc[(d.hypertension == 1) & (d.heart_disease == 1)
          & (d.avg_glucose_level > 200) & (d.age > 60), "risiko_nested"] = 4
    return f


TURUNAN = ["glukosa_kat", "bmi_kat", "jumlah_faktor", "risiko_nested"]
X_komb = pd.concat([X_dasar, fitur_turunan(df_penuh)], axis=1).astype(float)

print("Fitur dasar   :", X_dasar.shape[1])
print("Fitur turunan :", len(TURUNAN))
print("Total fitur   :", X_komb.shape[1])


Kelompok atribut dipetakan ke kolomnya. Sebuah kelompok kategorikal dicocokkan ke
kolom one-hot-nya, misalnya `gender` cocok dengan `gender_Male`.


In [ ]:
GRUP = {
    "usia (age)": ["age"],
    "hipertensi": ["hypertension"],
    "penyakit jantung": ["heart_disease"],
    "glukosa": ["avg_glucose_level"],
    "bmi": ["bmi"],
    "jenis kelamin": ["gender"],
    "status menikah": ["ever_married"],
    "pekerjaan": ["work_type"],
    "wilayah": ["Residence_type"],
    "merokok": ["smoking_status"],
}
KLINIS = ["usia (age)", "hipertensi", "penyakit jantung", "glukosa"]
SOSIAL = ["jenis kelamin", "status menikah", "pekerjaan", "wilayah", "merokok"]


def kolom(grup):
    """Ubah nama kelompok jadi daftar kolom.

    Kolom kategorikal dicocokkan ke kolom one-hot: 'gender' cocok dengan 'gender_Male'.
    Turunan if-else ditambahkan apa adanya. dict.fromkeys membuang kolom kembar.
    """
    kol = []
    for g in grup:
        if g == "turunan if-else":
            kol += TURUNAN
        else:
            for k in GRUP[g]:
                kol += [c for c in X_dasar.columns if c == k or c.startswith(k + "_")]
    return list(dict.fromkeys(kol))


Delapan kombinasi yang diadu, dari yang paling ramping sampai paling lengkap. Jumlah
fiturnya adalah jumlah kolom setelah one-hot dan turunan ikut dihitung.


In [ ]:
KOMBINASI = {
    "A. Usia saja": ["usia (age)"],
    "B. Usia + glukosa": ["usia (age)", "glukosa"],
    "C. Usia + glukosa + hipertensi + bmi": ["usia (age)", "glukosa", "hipertensi", "bmi"],
    "D. Usia + hipertensi + penyakit jantung": ["usia (age)", "hipertensi", "penyakit jantung"],
    "E. Inti klinis": KLINIS,
    "F. Inti klinis + bmi": KLINIS + ["bmi"],
    "G. Semua fitur dasar": KLINIS + ["bmi"] + SOSIAL,
    "H. Semua fitur + turunan if-else": KLINIS + ["bmi"] + SOSIAL + ["turunan if-else"],
}

ringkas = pd.DataFrame(
    [{"Kombinasi": nama, "jumlah fitur": len(kolom(grup))}
     for nama, grup in KOMBINASI.items()]
)
ringkas


### 6b. Alat ukur, semuanya dibuat manual

Tidak ada scikit-learn di sini, termasuk untuk pembagian lipatan dan perhitungan metrik.

- **Lipatan stratified manual.** Tiap kelas diacak sendiri lalu dibagi rata ke lima
  lipatan, supaya proporsi stroke tiap lipatan tetap.
- **Standardisasi manual.** Jarak wajib memakai fitur berskala sama. Mean dan simpangan
  baku dihitung dari data latih lipatan saja, lalu dipakai ke data ujinya, supaya tidak
  ada kebocoran.
- **Metrik manual.** AP (tie-aware, sama seperti scikit-learn), precision pada recall
  0,80, ROC-AUC, dan selang kepercayaan bootstrap 95% untuk AP.


In [ ]:
def lipatan_stratified(y, n_lipatan=5, seed=42):
    """Bagi indeks jadi n lipatan dengan proporsi kelas tiap lipatan tetap."""
    acak = np.random.default_rng(seed)
    lipatan = np.empty(len(y), dtype=int)
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        acak.shuffle(idx)
        lipatan[idx] = np.arange(len(idx)) % n_lipatan
    return lipatan


def standardisasi(X_latih, X_uji):
    """Standardisasi memakai mean dan std data latih saja."""
    mu = X_latih.mean(axis=0)
    sd = X_latih.std(axis=0)
    sd[sd == 0] = 1.0
    return (X_latih - mu) / sd, (X_uji - mu) / sd


Skor confidence Naive Bayes adalah **probabilitas posterior kelas stroke** dari model
Gaussian Naive Bayes manual. Modelnya memakai mean dan std per kelas, jadi tidak perlu
standardisasi, berbeda dari KNN.


In [ ]:
def proba_stroke(nb, X):
    """Probabilitas posterior kelas stroke dari Naive Bayes manual."""
    hasil = []
    for titik in X:
        log_prob = {}
        for c in nb.kelas:
            lp = math.log(nb.prior[c])
            for i in range(len(titik)):
                lp += math.log(gaussian_pdf(titik[i], nb.mean[c][i], nb.std[c][i]) + 1e-10)
            log_prob[c] = lp
        m = max(log_prob.values())
        total = sum(math.exp(v - m) for v in log_prob.values())
        hasil.append(math.exp(log_prob[1] - m) / total)
    return np.array(hasil)


def skor_oof(X_tab, y, n_lipatan=5, seed=42):
    """Skor out-of-fold Naive Bayes pada lipatan stratified."""
    lipatan = lipatan_stratified(y, n_lipatan, seed)
    skor = np.zeros(len(y))
    for f in range(n_lipatan):
        uji = lipatan == f
        nb = NaiveBayesManual()
        nb.fit(X_tab[~uji], y[~uji])
        skor[uji] = proba_stroke(nb, X_tab[uji])
    return skor


Skor KNN bernilai diskret, jadi kurva precision-recall dihitung pada tiap nilai skor
unik. Cara ini sama dengan yang dipakai scikit-learn, sehingga angka AP-nya bisa diadu
langsung dengan notebook `13`.


In [ ]:
def kurva_pr(y_true, skor):
    """Presisi dan recall pada tiap nilai skor unik (tie-aware, sama seperti scikit-learn)."""
    urut = np.argsort(-skor, kind="mergesort")
    yt = y_true[urut].astype(float)
    nilai = skor[urut]
    batas = np.r_[np.where(np.diff(nilai) != 0)[0], len(nilai) - 1]
    tp = np.cumsum(yt)[batas]
    fp = 1 + batas - tp
    if tp[-1] == 0:
        return np.array([0.0]), np.array([0.0])
    return tp / (tp + fp), tp / tp[-1]


def average_precision(y_true, skor):
    """Luas di bawah kurva precision-recall."""
    pres, rec = kurva_pr(y_true, skor)
    pres = np.r_[pres[::-1], 1.0]
    rec = np.r_[rec[::-1], 0.0]
    return float(-np.sum(np.diff(rec) * pres[:-1]))


def presisi_pada_recall(y_true, skor, target=0.80):
    """Precision terbesar yang masih mencapai recall >= target."""
    pres, rec = kurva_pr(y_true, skor)
    idx = np.where(rec >= target)[0]
    return float(pres[idx].max()) if len(idx) else float("nan")


def roc_auc(y_true, skor):
    """ROC-AUC manual lewat peringkat (statistik Mann-Whitney)."""
    urut = np.argsort(skor, kind="mergesort")
    peringkat = np.empty(len(skor), dtype=float)
    peringkat[urut] = np.arange(1, len(skor) + 1)
    nilai = skor[urut]
    i = 0
    while i < len(nilai):
        j = i
        while j + 1 < len(nilai) and nilai[j + 1] == nilai[i]:
            j += 1
        if j > i:
            peringkat[urut[i:j + 1]] = (i + j + 2) / 2
        i = j + 1
    pos = y_true == 1
    n1, n0 = pos.sum(), (~pos).sum()
    return float((peringkat[pos].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


def bootstrap_ap(y_true, skor, B=2000, seed=42):
    """Selang kepercayaan 95% untuk average precision."""
    acak = np.random.default_rng(seed)
    n = len(y_true)
    nilai = []
    for _ in range(B):
        idx = acak.integers(0, n, n)
        if y_true[idx].sum() == 0:
            continue
        nilai.append(average_precision(y_true[idx], skor[idx]))
    return float(np.percentile(nilai, 2.5)), float(np.percentile(nilai, 97.5))


### 6c. Hasil: confidence level tiap kombinasi

Semua kombinasi diukur dengan model dan protokol yang sama. Diurutkan dari AP terbesar.


In [ ]:
baris = []
for nama, grup in KOMBINASI.items():
    kol = kolom(grup)
    skor = skor_oof(X_komb[kol].to_numpy(), y_komb)
    ap = average_precision(y_komb, skor)
    lo, hi = bootstrap_ap(y_komb, skor)
    baris.append({
        "Kombinasi atribut": nama,
        "jumlah fitur": len(kol),
        "AP": round(ap, 4),
        "AP 2,5%": round(lo, 4),
        "AP 97,5%": round(hi, 4),
        "precision @ recall 0,80": round(presisi_pada_recall(y_komb, skor), 4),
        "ROC-AUC": round(roc_auc(y_komb, skor), 4),
    })

tabel_nb = pd.DataFrame(baris).sort_values("AP", ascending=False).reset_index(drop=True)
tabel_nb


Diperlihatkan sebagai gambar. Batang abu-abu tipis adalah selang kepercayaan 95% dari
AP, dan itu bagian terpentingnya: hampir semua selang bertindihan.


In [ ]:
urut = tabel_nb.sort_values("AP")
galat = np.array([urut["AP"] - urut["AP 2,5%"], urut["AP 97,5%"] - urut["AP"]])

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.barh(urut["Kombinasi atribut"], urut["AP"], xerr=galat, color="#4C78A8",
        error_kw={"ecolor": "#555", "capsize": 3})
ax.set_xlabel("Average precision (confidence level)")
ax.set_title('Naive Bayes manual: inti klinis tetap terbaik, selangnya bertindihan')
plt.tight_layout()
plt.show()


**Yang terbaca dari tabel.**

1. Kombinasi dengan AP terbaik adalah **inti klinis + bmi** (AP 0,1704), praktis seri
   dengan **inti klinis** (0,1702). Selisih 0,0002 jauh di dalam derau, jadi `bmi`
   tidak menambah apa pun setelah fitur lain ada.
2. Usia saja sudah membawa AP 0,1658. Menambah semua fitur dasar justru menurunkan AP
   ke 0,1490, dan menambah turunan if-else makin turun ke 0,1472. Jadi atribut tambahan
   **tidak** menambah confidence.
3. Selang kepercayaan semua kombinasi bertindihan lebar, jadi beda antar kombinasi besar
   **tidak boleh** diklaim bermakna.
4. Naive Bayes memilih inti klinis, sama seperti di notebook `13` (AP 0,166 di sana).
   Angka manual di sini sedikit lebih tinggi karena lipatannya dibangun sendiri.
5. Naive Bayes lebih baik dari KNN manual (AP tertinggi KNN 0,1443), tetapi masih di
   bawah Logistic Regression (0,184 di notebook `13`). Ini sejalan dengan temuan
   notebook `11` bahwa model sederhana menang pada data ini.


## 7. Kesimpulan

Gaussian Naive Bayes manual berhasil dijalankan dan dievaluasi, sepenuhnya tanpa
scikit-learn.

| Metrik | Nilai |
|---|---|
| Akurasi | 0,8727 |
| Presisi | 0,2149 |
| Recall | 0,4262 |
| F1 | 0,2857 |

Confusion matrix: TP=26, FP=95, FN=35, TN=865. Dari 61 pasien stroke di data uji, 26
berhasil terdeteksi.

### Dibandingkan aturan medis

Skor CHA2DS2-VASc di bagian 5 tidak dilatih sama sekali, tetapi hasilnya sepadan:

| Cara | Recall | Presisi | F1 |
|---|---|---|---|
| Skor medis, ambang skor >= 2 | 0,6885 | 0,1368 | 0,2283 |
| Skor medis, ambang skor >= 3 | 0,5082 | 0,2109 | 0,2981 |
| Naive Bayes | 0,4262 | 0,2149 | 0,2857 |

Naive Bayes hanya unggul tipis pada presisi (0,2149 lawan 0,2109). Pada recall dan F1 ia
justru di bawah ambang skor >= 3 (0,4262 lawan 0,5082, dan 0,2857 lawan 0,2981).
Kesimpulannya: untuk data sesederhana ini, **aturan medis yang tidak belajar apa pun
sepadan, bahkan sedikit lebih baik, dibanding Naive Bayes**. Itu alasan kuat untuk selalu
membandingkan model dengan cara sederhana sebelum mengklaim berhasil.

Catatan penting:

1. **Asumsi independensi antar fitur.** Naive Bayes menganggap tiap fitur saling bebas
   setelah diketahui kelasnya. Di data ini asumsi itu tidak sepenuhnya benar (usia dan
   hipertensi jelas berkorelasi), tetapi model tetap layak dipakai sebagai pembanding.
2. **Parameternya masuk akal secara medis.** Rata-rata usia kelas stroke 68,3 tahun
   lawan 41,8 tahun pada kelas tidak stroke, dan glukosanya 132,5 lawan 104,5. Model
   menangkap sinyal yang sejalan dengan pengetahuan medis.
3. **Tidak terpengaruh skala fitur** karena memakai mean dan std per kelas. Berbeda
   dari KNN di `03a_knn_manual.ipynb` yang wajib di-scaling.
4. **Recall masih jauh dari ideal.** Data hanya punya 4,87% kasus stroke, jadi model
   tetap cenderung menebak kelas mayoritas dan akurasi tetap menyesatkan.

### Dibandingkan uji kombinasi atribut

Uji kombinasi di bagian 6 menegaskan hal yang sama. Naive Bayes memberi confidence
terbaik pada inti klinis (usia, hipertensi, penyakit jantung, glukosa), dan menambah
atribut lain tidak menambah confidence. Ini sejalan dengan notebook `13` dan menegaskan
bahwa hanya segelintir atribut yang benar-benar membawa sinyal.

Langkah berikutnya: versi scikit-learn di `04_model_library.ipynb`.